# Task 1: Install required Python libraries

In [ ]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas
import sklearn
import streamlit
import joblib
import requests

print(pandas.__version__)
print(sklearn.__version__)
print(streamlit.__version__)
print(joblib.__version__)
print(requests.__version__)

# Task 2: Download the dataset

In [ ]:
from pathlib import Path
import requests
import pandas as pd

# Create data/ folder
Path('data').mkdir(exist_ok=True)

# Download results.csv only if it doesn't already exist
csv_path = Path('data/results.csv')
if not csv_path.exists():
    url = 'https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge/hands-on-labs/main/02_football_lab_june/04_data/results.csv'
    response = requests.get(url)
    response.raise_for_status()
    csv_path.write_bytes(response.content)

# Load into DataFrame
matches = pd.read_csv(csv_path, parse_dates=['date'])

print(matches.shape)
print(matches['date'].min())
print(matches['date'].max())
matches.head(3)

# Task 3: Explore the data

In [ ]:
# Top 10 most frequent tournaments
print('=== Top 10 Tournaments ===')
print(matches['tournament'].value_counts().head(10).to_string())

# Top 15 teams by total matches played (home + away)
print('\n=== Top 15 Teams by Matches Played ===')
team_counts = (
    pd.concat([matches['home_team'], matches['away_team']])
    .value_counts()
    .head(15)
)
print(team_counts.to_string())

# Matches per decade in chronological order
print('\n=== Matches per Decade ===')
decade_counts = (
    matches['date'].dt.year
    .floordiv(10)
    .mul(10)
    .value_counts()
    .sort_index()
    .rename(lambda y: f'{y}s')
)
print(decade_counts.to_string())

# Task 4: Engineer features for the model

In [ ]:
from collections import defaultdict

MAJOR_TOURNAMENTS = {
    'FIFA World Cup', 'FIFA World Cup qualification',
    'UEFA Euro', 'UEFA Euro qualification',
    'Copa América', 'African Cup of Nations'
}

# Helper functions
def winrate(hist):
    return sum(h[2] for h in hist) / len(hist) if hist else 0.5

def goal_avg(hist):
    return sum(h[0] for h in hist) / len(hist) if hist else 1.0

def recent_form(hist):
    last10 = hist[-10:]
    return sum(h[2] for h in last10) / len(last10) if len(last10) >= 10 else 0.5

# Filter and sort
filtered = (
    matches[matches['date'] >= '1990-01-01']
    .sort_values('date')
    .reset_index(drop=True)
)

team_history = defaultdict(list)  # team -> [(goals_for, goals_against, won), ...]
rows = []

for _, row in filtered.iterrows():
    home = row['home_team']
    away = row['away_team']
    hs = row['home_score']
    as_ = row['away_score']

    hist_a = team_history[home]
    hist_b = team_history[away]

    # Compute features from history BEFORE this match
    if hs > as_:
        outcome = 0
        home_won, away_won = 1, 0
    elif hs == as_:
        outcome = 1
        home_won, away_won = 0, 0
    else:
        outcome = 2
        home_won, away_won = 0, 1

    rows.append({
        'date': row['date'],
        'home_team': home,
        'away_team': away,
        'team_a_winrate': winrate(hist_a),
        'team_b_winrate': winrate(hist_b),
        'team_a_goal_avg': goal_avg(hist_a),
        'team_b_goal_avg': goal_avg(hist_b),
        'team_a_recent_form': recent_form(hist_a),
        'team_b_recent_form': recent_form(hist_b),
        'is_neutral': int(row['neutral']),
        'is_major_tournament': int(row['tournament'] in MAJOR_TOURNAMENTS),
        'outcome': outcome
    })

    # Append result to history AFTER computing features
    team_history[home].append((hs, as_, home_won))
    team_history[away].append((as_, hs, away_won))

features_df = pd.DataFrame(rows)

print(features_df.shape)
features_df.head(3)

# Task 5: Split data into training and test sets

In [ ]:
feature_cols = [
    'team_a_winrate', 'team_b_winrate',
    'team_a_goal_avg', 'team_b_goal_avg',
    'team_a_recent_form', 'team_b_recent_form',
    'is_neutral', 'is_major_tournament'
]

train = features_df[features_df['date'] < '2018-01-01']
test  = features_df[features_df['date'] >= '2018-01-01']

X_train = train[feature_cols]
y_train = train['outcome']
X_test  = test[feature_cols]
y_test  = test['outcome']

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test: ', X_test.shape)
print('y_test: ', y_test.shape)
print('\ny_train class distribution:')
print(y_train.value_counts(normalize=True).sort_index().to_string())

# Task 6: Train and evaluate the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

# Train
model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {accuracy * 100:.2f}%')

# Baseline: always predict most frequent class in y_train
most_frequent = y_train.value_counts().idxmax()
baseline = accuracy_score(y_test, [most_frequent] * len(y_test))
print(f'Baseline accuracy (always predict {most_frequent}): {baseline * 100:.2f}%')

# Confusion matrix
labels = ['Home win', 'Draw', 'Away win']
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f'Actual: {l}' for l in labels],
                          columns=[f'Predicted: {l}' for l in labels])
print('\nConfusion Matrix:')
print(cm_df.to_string())

# Feature importances
print('\nFeature importances (descending):')
importances = sorted(zip(feature_cols, model.feature_importances_),
                     key=lambda x: x[1], reverse=True)
for name, score in importances:
    print(f'  {name}: {score:.4f}')

# Task 7: Save the model and team statistics

In [ ]:
import joblib
from pathlib import Path

Path('models').mkdir(exist_ok=True)

# Teams eligible for the dropdown: must have appeared in World Cup qualification
wc_qual = matches[matches['tournament'] == 'Soccer World Cup qualification']
soccer_teams = set(wc_qual['home_team']) | set(wc_qual['away_team'])

# All teams in the full dataset
all_teams = set(matches['home_team']) | set(matches['away_team'])

team_stats = {}

for team in all_teams:
    # Skip non-World-Cup-eligible entities
    if team not in soccer_teams:
        continue

    home_matches = matches[matches['home_team'] == team]
    away_matches = matches[matches['away_team'] == team]
    total = len(home_matches) + len(away_matches)

    if total < 30:
        continue

    # Wins
    home_wins = (home_matches['home_score'] > home_matches['away_score']).sum()
    away_wins = (away_matches['away_score'] > away_matches['home_score']).sum()
    wins = home_wins + away_wins

    # Goals scored
    home_goals = home_matches['home_score'].sum()
    away_goals = away_matches['away_score'].sum()
    goal_avg = (home_goals + away_goals) / total

    # Recent form: last 10 matches by date
    home_df = home_matches[['date']].assign(goals=home_matches['home_score'].values,
                                             opp=home_matches['away_score'].values)
    away_df = away_matches[['date']].assign(goals=away_matches['away_score'].values,
                                             opp=away_matches['home_score'].values)
    all_m = pd.concat([home_df, away_df]).sort_values('date')
    last10 = all_m.tail(10)
    if len(last10) >= 10:
        recent = (last10['goals'] > last10['opp']).sum() / 10
    else:
        recent = 0.5

    team_stats[team] = {
        'winrate': wins / total,
        'goal_avg': goal_avg,
        'recent_form': float(recent),
        'matches_played': total
    }

# Save model and team data
joblib.dump(model, 'models/match_predictor.pkl')
joblib.dump({'team_stats': team_stats, 'feature_cols': feature_cols}, 'models/team_data.pkl')

print(f'Teams stored: {len(team_stats)}')

# Top 5 teams by winrate among those with >= 100 matches
top5 = sorted(
    [(t, s) for t, s in team_stats.items() if s['matches_played'] >= 100],
    key=lambda x: x[1]['winrate'],
    reverse=True
)[:5]
print('\nTop 5 teams by win rate (≥100 matches):')
for team, s in top5:
    print(f"  {team}: {s['winrate']:.3f} win rate over {s['matches_played']} matches")

# Task 8: Try the model

In [ ]:
import pandas as pd

def predict_match(team_a, team_b, is_neutral=True, is_major_tournament=True):
    if team_a not in team_stats:
        raise ValueError(f"'{team_a}' not found in team_stats. Check spelling or use a team with ≥30 matches.")
    if team_b not in team_stats:
        raise ValueError(f"'{team_b}' not found in team_stats. Check spelling or use a team with ≥30 matches.")

    a = team_stats[team_a]
    b = team_stats[team_b]

    row = pd.DataFrame([{
        'team_a_winrate':     a['winrate'],
        'team_b_winrate':     b['winrate'],
        'team_a_goal_avg':    a['goal_avg'],
        'team_b_goal_avg':    b['goal_avg'],
        'team_a_recent_form': a['recent_form'],
        'team_b_recent_form': b['recent_form'],
        'is_neutral':         int(is_neutral),
        'is_major_tournament': int(is_major_tournament),
    }])[feature_cols]  # guarantee column order matches training

    proba = model.predict_proba(row)
    return {
        'team_a_win_prob': round(float(proba[0][0]), 4),
        'draw_prob':       round(float(proba[0][1]), 4),
        'team_b_win_prob': round(float(proba[0][2]), 4),
    }

result1 = predict_match('Brazil', 'Argentina')
print('Brazil vs Argentina:')
print(f"  Brazil win:    {result1['team_a_win_prob']:.1%}")
print(f"  Draw:          {result1['draw_prob']:.1%}")
print(f"  Argentina win: {result1['team_b_win_prob']:.1%}")

print()

result2 = predict_match('Germany', 'Brazil')
print('Germany vs Brazil:')
print(f"  Germany win: {result2['team_a_win_prob']:.1%}")
print(f"  Draw:        {result2['draw_prob']:.1%}")
print(f"  Brazil win:  {result2['team_b_win_prob']:.1%}")

# Task 9: Create an interactive UI application

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

st.set_page_config(
    page_title='Soccer 2026 Match Predictor',
    page_icon='⚽',
    layout='centered'
)

@st.cache_resource
def load_artifacts():
    model = joblib.load(Path('models/match_predictor.pkl'))
    team_data = joblib.load(Path('models/team_data.pkl'))
    return model, team_data['team_stats'], team_data['feature_cols']

model, team_stats, feature_cols = load_artifacts()

st.title('⚽ Soccer 2026 Match Predictor')
st.caption('Predictions based on historical international football results (1872–2026).')

team_names = sorted(team_stats.keys())

col1, col2 = st.columns(2)
with col1:
    default_a = team_names.index('Brazil') if 'Brazil' in team_names else 0
    team_a = st.selectbox('Team A', team_names, index=default_a)
with col2:
    default_b = team_names.index('Argentina') if 'Argentina' in team_names else 1
    team_b = st.selectbox('Team B', team_names, index=default_b)

is_neutral = st.checkbox('Neutral venue', value=True)
is_major   = st.checkbox('Major tournament (e.g. World Cup)', value=True)

if st.button('Predict', type='primary', use_container_width=True):
    if team_a == team_b:
        st.error('Please pick two different teams.')
    else:
        a = team_stats[team_a]
        b = team_stats[team_b]

        row = pd.DataFrame([{
            'team_a_winrate':      a['winrate'],
            'team_b_winrate':      b['winrate'],
            'team_a_goal_avg':     a['goal_avg'],
            'team_b_goal_avg':     b['goal_avg'],
            'team_a_recent_form':  a['recent_form'],
            'team_b_recent_form':  b['recent_form'],
            'is_neutral':          int(is_neutral),
            'is_major_tournament': int(is_major),
        }])[feature_cols]

        proba = model.predict_proba(row)[0]
        p_a, p_draw, p_b = float(proba[0]), float(proba[1]), float(proba[2])

        st.subheader('Prediction')
        m1, m2, m3 = st.columns(3)
        m1.metric(f'{team_a} wins', f'{p_a:.1%}')
        m2.metric('Draw',           f'{p_draw:.1%}')
        m3.metric(f'{team_b} wins', f'{p_b:.1%}')

        st.write('')
        st.progress(p_a,    text=f'{team_a} wins: {p_a:.1%}')
        st.progress(p_draw, text=f'Draw: {p_draw:.1%}')
        st.progress(p_b,    text=f'{team_b} wins: {p_b:.1%}')

        st.subheader('Team statistics')
        stats_table = pd.DataFrame({
            'Team':           [team_a, team_b],
            'Win rate':       [f"{a['winrate']:.3f}",      f"{b['winrate']:.3f}"],
            'Avg goals':      [f"{a['goal_avg']:.2f}",     f"{b['goal_avg']:.2f}"],
            'Recent form':    [f"{a['recent_form']:.3f}",  f"{b['recent_form']:.3f}"],
            'Matches played': [a['matches_played'],         b['matches_played']],
        }).set_index('Team')
        st.table(stats_table)

# Task 10: Launch the UI application

In [33]:
import subprocess
import sys
import time
import webbrowser

streamlit_process = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', 'app.py',
        '--server.headless', 'true',
        '--server.port', '8501',
        '--browser.gatherUsageStats', 'false'
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print('Starting Streamlit server...')
time.sleep(4)

webbrowser.open('http://localhost:8501')

print('Streamlit app is running at http://localhost:8501')
print('To stop the server, run streamlit_process.terminate() in a new cell.')

Starting Streamlit server...
Streamlit app is running at http://localhost:8501
To stop the server, run streamlit_process.terminate() in a new cell.


# Task 11: Enhance the app with Team History and Rules of the Game

In [34]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import requests
from pathlib import Path

st.set_page_config(
    page_title='Soccer 2026 Match Predictor',
    page_icon='⚽',
    layout='centered'
)

@st.cache_resource
def load_artifacts():
    model = joblib.load(Path('models/match_predictor.pkl'))
    team_data = joblib.load(Path('models/team_data.pkl'))
    return model, team_data['team_stats'], team_data['feature_cols']

@st.cache_data
def load_matches():
    return pd.read_csv(Path('data/results.csv'), parse_dates=['date'])

model, team_stats, feature_cols = load_artifacts()
matches_df = load_matches()

st.title('⚽ Soccer 2026 Match Predictor')
st.caption('Predictions based on historical international football results (1872–2026).')

team_names = sorted(team_stats.keys())

tab1, tab2, tab3 = st.tabs(['⚽ Predictor', '📖 Team History', '📋 Rules of the Game'])

with tab1:
    col1, col2 = st.columns(2)
    with col1:
        default_a = team_names.index('Brazil') if 'Brazil' in team_names else 0
        team_a = st.selectbox('Team A', team_names, index=default_a)
    with col2:
        default_b = team_names.index('Argentina') if 'Argentina' in team_names else 1
        team_b = st.selectbox('Team B', team_names, index=default_b)

    is_neutral = st.checkbox('Neutral venue', value=True)
    is_major   = st.checkbox('Major tournament (e.g. World Cup)', value=True)

    if st.button('Predict', type='primary', use_container_width=True):
        if team_a == team_b:
            st.error('Please pick two different teams.')
        else:
            a = team_stats[team_a]
            b = team_stats[team_b]

            row = pd.DataFrame([{
                'team_a_winrate':      a['winrate'],
                'team_b_winrate':      b['winrate'],
                'team_a_goal_avg':     a['goal_avg'],
                'team_b_goal_avg':     b['goal_avg'],
                'team_a_recent_form':  a['recent_form'],
                'team_b_recent_form':  b['recent_form'],
                'is_neutral':          int(is_neutral),
                'is_major_tournament': int(is_major),
            }])[feature_cols]

            proba = model.predict_proba(row)[0]
            p_a, p_draw, p_b = float(proba[0]), float(proba[1]), float(proba[2])

            st.subheader('Prediction')
            m1, m2, m3 = st.columns(3)
            m1.metric(f'{team_a} wins', f'{p_a:.1%}')
            m2.metric('Draw',           f'{p_draw:.1%}')
            m3.metric(f'{team_b} wins', f'{p_b:.1%}')

            st.write('')
            st.progress(p_a,    text=f'{team_a} wins: {p_a:.1%}')
            st.progress(p_draw, text=f'Draw: {p_draw:.1%}')
            st.progress(p_b,    text=f'{team_b} wins: {p_b:.1%}')

            st.subheader('Team statistics')
            stats_table = pd.DataFrame({
                'Team':           [team_a, team_b],
                'Win rate':       [f"{a['winrate']:.3f}",      f"{b['winrate']:.3f}"],
                'Avg goals':      [f"{a['goal_avg']:.2f}",     f"{b['goal_avg']:.2f}"],
                'Recent form':    [f"{a['recent_form']:.3f}",  f"{b['recent_form']:.3f}"],
                'Matches played': [a['matches_played'],         b['matches_played']],
            }).set_index('Team')
            st.table(stats_table)

@st.cache_data(ttl=3600)
def fetch_wikipedia_bio(team_name):
    try:
        title = f"{team_name} national football team"
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(title)}"
        resp = requests.get(url, timeout=5)
        if resp.status_code == 200:
            return resp.json().get('extract')
        return None
    except Exception:
        return None

with tab2:
    selected_team = st.selectbox('Select a team', team_names)

    team_matches = matches_df[
        (matches_df['home_team'] == selected_team) |
        (matches_df['away_team'] == selected_team)
    ].copy()

    def classify(row, team):
        if row['home_team'] == team:
            scored, conceded = row['home_score'], row['away_score']
        else:
            scored, conceded = row['away_score'], row['home_score']
        if scored > conceded:
            return 'W', scored, conceded
        elif scored == conceded:
            return 'D', scored, conceded
        else:
            return 'L', scored, conceded

    results_data = team_matches.apply(lambda r: classify(r, selected_team), axis=1)
    team_matches['_result']   = results_data.apply(lambda x: x[0])
    team_matches['_scored']   = results_data.apply(lambda x: x[1])
    team_matches['_conceded'] = results_data.apply(lambda x: x[2])

    wins   = (team_matches['_result'] == 'W').sum()
    draws  = (team_matches['_result'] == 'D').sum()
    losses = (team_matches['_result'] == 'L').sum()
    goals_scored   = int(team_matches['_scored'].sum())
    goals_conceded = int(team_matches['_conceded'].sum())

    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric('Wins',           wins)
    c2.metric('Draws',          draws)
    c3.metric('Losses',         losses)
    c4.metric('Goals Scored',   goals_scored)
    c5.metric('Goals Conceded', goals_conceded)

    st.subheader('Last 5 Results')
    last5 = team_matches.sort_values('date', ascending=False).head(5)

    def build_last5_row(row, team):
        if row['home_team'] == team:
            opponent = row['away_team']
            score    = f"{int(row['home_score'])} – {int(row['away_score'])}"
        else:
            opponent = row['home_team']
            score    = f"{int(row['away_score'])} – {int(row['home_score'])}"
        return pd.Series({
            'Date':     row['date'].strftime('%Y-%m-%d'),
            'Opponent': opponent,
            'Score':    score,
            'Result':   row['_result'],
        })

    last5_display = last5.apply(lambda r: build_last5_row(r, selected_team), axis=1).reset_index(drop=True)
    st.table(last5_display)

    st.subheader(f'About {selected_team}')
    bio = fetch_wikipedia_bio(selected_team)
    if bio:
        st.write(bio)
    else:
        st.info('No biography available for this team.')

with tab3:
    st.write('New to football? Here are the essential rules of the game.')

    with st.expander('🎯 The Objective'):
        st.markdown("""
- The aim is to score more goals than the opposing team by the end of the match.
- A goal is scored when the whole ball crosses the goal line between the posts and under the crossbar.
- The team with the most goals at the end wins. If scores are level it is a draw (in group stages) or the game may go to extra time/penalties (knockout rounds).
        """)

    with st.expander('👥 Players & Positions'):
        st.markdown("""
- Each team fields **11 players**, including one designated **goalkeeper**.
- Common outfield positions: **defenders** (protect their goal), **midfielders** (link defence and attack), **forwards/strikers** (score goals).
- Teams may make up to **5 substitutions** per match (3 in some competitions).
- A team reduced to fewer than 7 players must forfeit the match.
        """)

    with st.expander('⏱️ Match Duration'):
        st.markdown("""
- A standard match consists of **two 45-minute halves** with a 15-minute half-time break.
- The referee adds **stoppage time** at the end of each half to compensate for injuries, substitutions, and delays.
- In knockout competitions, if the score is level after 90 minutes, **extra time** (two 15-minute periods) is played.
- If still level after extra time, the match is decided by a **penalty shootout**.
        """)

    with st.expander('⚽ Scoring'):
        st.markdown("""
- A goal counts when the **entire ball** crosses the goal line between the goalposts and under the crossbar.
- **Own goals** (accidentally scored by a defending player) count for the attacking team.
- Goals scored in open play, from free kicks, penalties, and corners all count equally.
- The Video Assistant Referee (**VAR**) may review goals to check for offside or handball.
        """)

    with st.expander('🔄 Restarts'):
        st.markdown("""
- **Kick-off**: starts each half and restarts play after a goal.
- **Throw-in**: awarded when the ball goes out of play over the sideline — thrown in by the opposing team to the one that last touched it.
- **Goal kick**: awarded to the defending team when the ball crosses the goal line last touched by an attacker.
- **Corner kick**: awarded to the attacking team when the ball crosses the goal line last touched by a defender.
- **Free kick**: awarded after a foul — can be **direct** (shot straight at goal) or **indirect** (must touch another player first).
        """)

    with st.expander('🚩 The Offside Rule'):
        st.markdown("""
- A player is in an **offside position** if they are nearer to the opponent's goal line than both the ball and the **second-to-last defender** at the moment the ball is played to them.
- Being offside is **not an offence by itself** — it only becomes one if the player is actively involved in play.
- A player **cannot be offside** from a goal kick, throw-in, or corner kick.
- VAR uses a line system to check marginal offside calls.
        """)

    with st.expander('🟨🟥 Fouls & Cards'):
        st.markdown("""
- A **foul** is called when a player trips, pushes, holds, or handles the ball illegally.
- **Yellow card** = a formal warning. Two yellow cards in the same match = automatic red card.
- **Red card** = the player is immediately sent off and cannot be replaced — their team plays with 10 players.
- A **straight red** can be given for serious foul play, violent conduct, or denying an obvious goal-scoring opportunity.
- Fouls inside the penalty area result in a **penalty kick**.
        """)

    with st.expander('🥅 Penalty Shootout'):
        st.markdown("""
- Used in **knockout rounds** when the score is still level after extra time.
- Each team nominates 5 players to take turns shooting from the **penalty spot** (11 metres from goal).
- The team that scores the most of their 5 penalties wins.
- If still level after 5 each, it goes to **sudden death** — teams alternate one penalty at a time until one scores and the other misses.
- The goalkeeper must stay on their line until the ball is kicked.
        """)

Overwriting app.py
